In [ ]:
import numpy as np
from scipy.integrate import RK45
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.collections import LineCollection
import matplotlib.colors as mcolors 
from collections import deque

# Constants

In [ ]:
g = 9.81 # acceleration due to gravity on Earth (ms^-1)

# Derivatives



In [ ]:
def derivatives(t, state, mass, length):
    angle_1, angular_velocity_1, angle_2, angular_velocity_2 = state
    mass_1, mass_2 = mass
    length_1, length_2 = length

    delta = angle_1 - angle_2
    sin_delta = np.sin(delta)
    cos_delta = np.cos(delta)

    derivative_1 = (
        -mass_2 * length_1 * (angular_velocity_1 ** 2) * sin_delta * cos_delta +
        mass_2 * g * np.sin(angle_2) * cos_delta -
        mass_2 * length_2 * (angular_velocity_2 ** 2) * sin_delta -
        (mass_1 + mass_2) * g * np.sin(angle_1)
    )/((mass_1 + mass_2) * length_1 - mass_2 * length_1 * (cos_delta ** 2))

    derivative_2 = (
        (mass_1 + mass_2) * length_1 * (angular_velocity_1 ** 2) * sin_delta -
        (mass_1 + mass_2) * g * np.sin(angle_2) +
        (mass_1 + mass_2) * g * np.sin(angle_1) * cos_delta + 
        mass_2 * length_2 * (angular_velocity_2 ** 2) * sin_delta * cos_delta
    )/((mass_1 + mass_2) * length_2 - mass_2 * length_2 * (cos_delta ** 2))

    return np.array([angular_velocity_1, derivative_1, angular_velocity_2, derivative_2])

# Pendulum Physics

In [ ]:
dt = 0.01 # step size

number_of_pendulums = 2
length = [[2, 1] for i in range(number_of_pendulums)]
mass = [[2, 1] for i in range(number_of_pendulums)]

initial_angle = np.array([170, 170])

initial_angle = np.array([initial_angle + (i * 0.0001) for i in range(number_of_pendulums)])

In [ ]:
solvers = [
     RK45(
        fun=lambda t, state, m=mass[i], l=length[i]: derivatives(t, state, m, l),
        t0=0,
        y0=[np.radians(initial_angle[i][0]), 0, np.radians(initial_angle[i][1]), 0],
        t_bound=np.inf,
        max_step=dt
    )
    for i in range(number_of_pendulums)
]

# Simulation

In [ ]:
%matplotlib widget

mfig, ax = plt.subplots()

# hide matplotlib widget ui
mfig.canvas.toolbar_visible = False
mfig.canvas.header_visible = False
mfig.canvas.footer_visible = False

PENDULUM_1_HEXs = ["#3F3F3F" for _ in range(number_of_pendulums)]
PALETTE = ["#ff0059", "#0091FF", "#00ff5e", "#9d00ff", "#ffaa00"]
PENDULUM_2_HEXs = [PALETTE[i % len(PALETTE)] for i in range(number_of_pendulums)]

# black background colour
ax.set_facecolor('black')
mfig.patch.set_facecolor('black')

# hide axes
ax.set_xticks([])
ax.set_yticks([])
ax.spines['top'].set_visible(False)
ax.spines['bottom'].set_visible(False)
ax.spines['left'].set_visible(False)
ax.spines['right'].set_visible(False)

total_length = max(np.sum(l) for l in length)

# set x and y axes to fit pendulum
ax.set_xlim(-(total_length * 1.2), total_length * 1.2)
ax.set_ylim(-(total_length * 1.2), total_length * 1.2)
ax.set_aspect("equal")

# pendulum 2 trace
TRACE_LENGTH = 100
trace_xs = [deque(maxlen=TRACE_LENGTH) for _ in range(number_of_pendulums)]
trace_ys = [deque(maxlen=TRACE_LENGTH) for i in range(number_of_pendulums)]
trace_collections = [LineCollection([], lw=1, color=PENDULUM_2_HEXs[i]) for i in range(number_of_pendulums)]

for trace_collection in trace_collections:
    ax.add_collection(trace_collection)

# pendulum 1
initial_position_1s = [length[i][0] * np.array([np.sin(solvers[i].y[0]), -np.cos(solvers[i].y[0])]) for i in range(number_of_pendulums)]
string_1s = [ax.plot([0, initial_position_1s[i][0]], [0, initial_position_1s[i][1]], "o-", linewidth=2.0, color=PENDULUM_1_HEXs[i])[0] for i in range(number_of_pendulums)]

# pendulum 2
initial_position_2s = [initial_position_1s[i] + (length[i] * np.array([np.sin(solvers[i].y[2]), -np.cos(solvers[i].y[2])]))  for i in range(number_of_pendulums)]
string_2s = [ax.plot([initial_position_1s[i][0], initial_position_2s[i][0]], [initial_position_1s[i][1], initial_position_2s[i][1]], "o-", linewidth=2.0, color=PENDULUM_2_HEXs[i])[0] for i in range(number_of_pendulums)]


def update(frame):
    for i in range(number_of_pendulums):
        # step solver forward by dt
        solvers[i].step()
        state = solvers[i].y

        # pendulum 1
        new_position_1 = length[i][0] * np.array([np.sin(state[0]), -np.cos(state[0])])
        string_1s[i].set_data([0, new_position_1[0]], [0, new_position_1[1]])

        # pendulum 2
        new_position_2 = new_position_1 + (length[i][1] * np.array([np.sin(state[2]), -np.cos(state[2])])) 
        string_2s[i].set_data([new_position_1[0], new_position_2[0]], [new_position_1[1], new_position_2[1]])
        
        # update trace to pendulum 2 new position
        trace_xs[i].append(new_position_2[0])
        trace_ys[i].append(new_position_2[1])

        tx = list(trace_xs[i])
        ty = list(trace_ys[i])
        n = len(tx)

        # fade trace over time
        if n > 1:
            points = np.array([tx, ty]).T.reshape(-1, 1, 2)
            segments = np.concatenate([points[:-1], points[1:]], axis=1)
            
            red, green, blue = mcolors.to_rgb(PENDULUM_2_HEXs[i])
            
            alphas = np.linspace(0, 0.8, len(segments))
            colors = [(red, green, blue, a) for a in alphas]
            
            trace_collections[i].set_colors(colors)
            trace_collections[i].set_segments(segments)
        
    return *string_1s, *string_2s, *trace_collections

anim = animation.FuncAnimation(mfig, update, interval=dt*1000, cache_frame_data=False)

plt.show()